# Memory Storage Technologies

**Module:** 13 — AI Memory

Vector DBs, SQL/KV, and graph memory — trade-offs and selection.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare vector databases, traditional databases, and graph stores for agent memory
- Sketch schemas/metadata for each approach
- Choose storage based on query patterns, consistency, and ops cost
- Design a hybrid (polyglot) memory architecture


## Vector Databases

### Definition
Stores embedding vectors plus metadata for approximate nearest-neighbor (ANN) similarity search — the workhorse of semantic memory and RAG.

### Why it matters
Natural language recall ('things like this') needs similarity, not only exact keys. Vector DBs make that query cheap at scale.

### How it works
Embed text → upsert `(id, vector, payload)` → query with a vector + metadata filter → rerank. Popular systems: Qdrant, Pinecone, Weaviate, pgvector, Chroma.

### Intuition
A vector DB is a library organized by *meaning neighborhood*, not alphabet.

### Pitfalls
- No metadata filters → wrong tenant / wrong project
- Stale embeddings after content edits
- Over-trusting ANN (approximate) for exact ID lookups

### When to use
Semantic preferences, document chunks, episode summaries, multimodal embeddings.


### Vector memory record (conceptual schema)

| Field | Type | Purpose |
|-------|------|---------|
| `id` | string | Stable identity |
| `vector` | float[d] | Embedding |
| `text` | string | Original or canonical text |
| `user_id` / `tenant_id` | string | Isolation |
| `memory_type` | enum | fact/episode/... |
| `created_at` | timestamp | Recency |
| `importance` | float | Ranking prior |

```mermaid
flowchart LR
  T[Text] --> E[Embed API]
  E --> U[Upsert ANN index]
  Q[Query text] --> E2[Embed]
  E2 --> S[Similarity search + filters]
  U --> S
  S --> R[Rerank / pack]
```


In [ ]:
# Demo 1: in-process ANN-ish store (brute force) with metadata filter
import math, os, json
from dataclasses import dataclass

@dataclass
class VecRow:
    id: str
    vector: list[float]
    text: str
    tenant_id: str
    memory_type: str

class BruteVectorDB:
    def __init__(self):
        self.rows: list[VecRow] = []

    def upsert(self, row: VecRow) -> None:
        self.rows = [r for r in self.rows if r.id != row.id] + [row]

    def search(self, vector, tenant_id: str, top_k: int = 3):
        def cos(a, b):
            dot = sum(x * y for x, y in zip(a, b))
            na = math.sqrt(sum(x * x for x in a)) or 1.0
            nb = math.sqrt(sum(x * x for x in b)) or 1.0
            return dot / (na * nb)
        scored = [
            (cos(vector, r.vector), r)
            for r in self.rows
            if r.tenant_id == tenant_id
        ]
        scored.sort(reverse=True)
        return scored[:top_k]

# Fake 4-d embeddings for teaching
db = BruteVectorDB()
db.upsert(VecRow("1", [1, 0, 0, 0], "likes dark mode", "t1", "preference"))
db.upsert(VecRow("2", [0.9, 0.1, 0, 0], "prefers dark themes", "t1", "preference"))
db.upsert(VecRow("3", [0, 1, 0, 0], "uses Postgres", "t1", "fact"))
db.upsert(VecRow("x", [1, 0, 0, 0], "LEAK", "t2", "preference"))
for score, row in db.search([1, 0, 0, 0], tenant_id="t1"):
    print(round(score, 3), row.text)


In [ ]:
# Demo 2: realistic Qdrant-like upsert/search request shapes (placeholders)
import os, json
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "YOUR_QDRANT_API_KEY")

upsert_body = {
    "points": [{
        "id": 42,
        "vector": [0.01, 0.02, 0.03],  # truncated example
        "payload": {
            "text": "User prefers UTC timestamps",
            "tenant_id": "t1",
            "memory_type": "preference",
        },
    }]
}
search_body = {
    "vector": [0.01, 0.02, 0.03],
    "limit": 5,
    "filter": {"must": [{"key": "tenant_id", "match": {"value": "t1"}}]},
}
print("endpoint", f"{QDRANT_URL}/collections/memories/points")
print(json.dumps({"upsert": upsert_body, "search": search_body}, indent=2)[:600])
print("api key placeholder?", QDRANT_API_KEY.startswith("YOUR_"))


### Try it yourself — Vectors

1. Add `memory_type` filter to `BruteVectorDB.search`.
2. Implement update-in-place when the same `id` is upserted with new text/vector.


## Traditional Databases

### Definition
Relational (Postgres/MySQL) and document/KV stores (Redis, DynamoDB, MongoDB) for exact lookups, transactions, and structured queries.

### Why it matters
Many memories are keyed facts ('theme=dark') or require joins, constraints, and ACID updates — vectors alone are a poor fit.

### How it works
Model entities and facts as tables/collections; use Redis for hot STM; use SQL for profiles, audit logs, and strong consistency.

### Intuition
When you know the *key* or need a transaction, use a traditional DB.

### Pitfalls
- Using SQL LIKE as a substitute for semantic search
- No indexes on `(tenant_id, user_id)`
- STM in Redis without TTL → memory leak

### When to use
Preferences, accounts, entitlements, audit trails, session state.


In [ ]:
# Demo 3: SQL-shaped schema in sqlite for durable facts
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE memory_facts (
  id INTEGER PRIMARY KEY,
  tenant_id TEXT NOT NULL,
  user_id TEXT NOT NULL,
  key TEXT NOT NULL,
  value TEXT NOT NULL,
  updated_at TEXT NOT NULL,
  UNIQUE(tenant_id, user_id, key)
);
""")
conn.execute(
    "INSERT INTO memory_facts(tenant_id,user_id,key,value,updated_at) VALUES (?,?,?,?,?)",
    ("t1", "u_42", "theme", "dark", "2026-08-01T00:00:00Z"),
)
conn.execute(
    """INSERT INTO memory_facts(tenant_id,user_id,key,value,updated_at)
       VALUES (?,?,?,?,?)
       ON CONFLICT(tenant_id,user_id,key) DO UPDATE SET value=excluded.value""",
    ("t1", "u_42", "theme", "dark-hc", "2026-08-01T12:00:00Z"),
)
print(conn.execute("SELECT user_id,key,value FROM memory_facts").fetchall())


In [ ]:
# Demo 4: Redis-like STM with TTL semantics (dict mock)
import time

class FakeRedis:
    def __init__(self):
        self._kv = {}  # key -> (value, expires_at|None)

    def set(self, key, value, ex=None):
        exp = time.time() + ex if ex is not None else None
        self._kv[key] = (value, exp)

    def get(self, key):
        item = self._kv.get(key)
        if not item:
            return None
        value, exp = item
        if exp is not None and time.time() > exp:
            del self._kv[key]
            return None
        return value

r = FakeRedis()
r.set("stm:u_42", '["hi","hello"]', ex=1)
print("immediate", r.get("stm:u_42"))
time.sleep(1.1)
print("after ttl", r.get("stm:u_42"))


## Graph Memory

### Definition
Stores entities and relationships (nodes/edges) for multi-hop reasoning: *User —PREFERS→ Theme*, *Project —USES→ Database*.

### Why it matters
Many agent questions are relational ('who owns the service that depends on Redis?'). Graphs shine at traversal and provenance.

### How it works
Extract entities/relations (LLM or rules) → upsert to Neo4j/Memgraph/NetworkX → query with Cypher/pattern matching; optionally embed node text too.

### Intuition
A mind map with typed arrows — great when relationships matter more than paragraphs.

### Pitfalls
- Over-extracting noisy edges from chat
- No tenant partitioning in the graph
- Using a graph when a simple KV fact would do

### When to use
Org knowledge, entity-centric assistants, multi-hop tooling maps, citation graphs.


In [ ]:
# Demo 5: tiny graph memory with NetworkX-style dicts
from collections import defaultdict

class GraphMemory:
    def __init__(self):
        self.nodes = {}  # id -> attrs
        self.edges = defaultdict(list)  # src -> list[(dst, rel, attrs)]

    def add_node(self, node_id: str, **attrs):
        self.nodes[node_id] = {**self.nodes.get(node_id, {}), **attrs}

    def add_edge(self, src: str, rel: str, dst: str, **attrs):
        self.edges[src].append((dst, rel, attrs))

    def neighbors(self, src: str, rel: str | None = None):
        out = []
        for dst, r, attrs in self.edges.get(src, []):
            if rel is None or r == rel:
                out.append((dst, r, attrs))
        return out

g = GraphMemory()
g.add_node("user:u_42", type="user")
g.add_node("proj:phoenix", type="project")
g.add_node("db:postgres", type="technology")
g.add_edge("user:u_42", "WORKS_ON", "proj:phoenix")
g.add_edge("proj:phoenix", "USES", "db:postgres", since="2026-01")
print("user projects:", g.neighbors("user:u_42", "WORKS_ON"))
print("phoenix tech:", g.neighbors("proj:phoenix", "USES"))


## Choosing Storage

| Query pattern | Prefer | Why |
|---------------|--------|-----|
| Exact key / transactional update | SQL / KV | Consistency, constraints |
| "Similar to this text" | Vector DB | ANN semantic search |
| Multi-hop entity relations | Graph | Traversal |
| Hot session buffer | Redis | TTL + speed |
| Mixed production agent | **Hybrid** | Polyglot persistence |

### Reference hybrid architecture
```
STM: Redis (TTL)
Facts: Postgres
Semantic: Qdrant/pgvector
Episodes: Postgres + vectors
Procedures: Git/YAML + object store
Entities: Graph (optional)
```


In [ ]:
# Demo 6: router that picks a backend by memory_type
def pick_backend(memory_type: str) -> str:
    return {
        "preference": "postgres",
        "fact": "postgres+vector",
        "episode": "vector+postgres",
        "procedural": "git_yaml",
        "session": "redis",
        "entity_link": "graph",
    }.get(memory_type, "postgres+vector")

for t in ["preference", "episode", "session", "entity_link", "unknown"]:
    print(t, "->", pick_backend(t))


### Try it yourself — Polyglot design

1. Draw (ASCII) a storage diagram for a multi-tenant support agent.
2. Write a migration note: how you would re-embed all facts after changing embedding models.

**Stretch:** Add a `freshness_hours` field and filter out stale tool-result memories.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `ANN` | Approximate nearest neighbor search over vectors |
| `payload/metadata` | Non-vector fields used for filtering |
| `pgvector` | Postgres extension for vector similarity |
| `polyglot persistence` | Multiple specialized stores in one system |
| `Cypher` | Common graph query language (Neo4j) |


## Ops Concerns by Store

| Store | Backup | Resize pain | Multi-tenant tactic |
|-------|--------|-------------|---------------------|
| Vector DB | Snapshots + replay upserts | Re-shard / re-embed | payload filter + separate collections |
| Postgres | PITR | Migrations | Row-level `tenant_id` |
| Redis | AOF/RDB | Memory ceiling | Key prefixes + TTL |
| Graph | Export subgraph | Reindex | Partition key / separate DB |

```mermaid
flowchart TB
  App --> API[Memory service]
  API --> PG[(Postgres facts)]
  API --> V[(Vector ANN)]
  API --> R[(Redis STM)]
  API --> G[(Graph optional)]
```


In [ ]:
# Collection-per-tenant vs shared-collection trade-off calculator
def estimate(tenants: int, mode: str, qps_per_tenant: float) -> dict:
    if mode == "shared":
        return {"collections": 1, "filter_overhead": "yes", "qps": tenants * qps_per_tenant}
    return {"collections": tenants, "filter_overhead": "no", "qps": qps_per_tenant}

print(estimate(200, "shared", 2))
print(estimate(200, "per_tenant", 2))


In [ ]:
# Re-embed migration sketch
def reembed_plan(docs: list[str], batch=2):
    batches = [docs[i:i+batch] for i in range(0, len(docs), batch)]
    return {
        "batches": len(batches),
        "dual_write": "write new_vector_field alongside old",
        "cutover": "flip search to new field after verify recall@k",
        "sample_first_batch": batches[0],
    }

print(reembed_plan([f"doc-{i}" for i in range(5)]))


### Try it yourself — Storage deepen

1. Propose when to use pgvector-only vs dedicated ANN service.
2. Write a Cypher-like pseudocode query for: projects user U works on that USE database X.


## Key Takeaways

- Match storage to query pattern — do not force everything into vectors
- Metadata filters and tenant keys are mandatory in vector stores
- SQL still wins for upsertable profile facts and audit
- Graphs help when relationships are first-class
- Production memory is usually hybrid
